In [1]:
from pathlib import Path

from maite_datasets.image_classification import MilitaryVehicles

data_root = Path("./data")

# One download shared by every export; a re-run reads what is already on disk.
MilitaryVehicles(root=data_root, image_set="base", download=True)
MilitaryVehicles(root=data_root, image_set="train", as_datamaite=True)

# The export nests its images one level down, under the split name.
data_path = data_root / "militaryvehicles_datamaite_train" / "train"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles_datamaite_train/train


In [2]:
from dataeval_flow import run_task
from dataeval_flow.config import HuggingFaceDatasetConfig, PipelineConfig, SourceConfig, TaskConfig
from dataeval_flow.workflows.bias import BiasConfig
from dataeval_flow.workflows.splits import SplitsConfig

workflow = SplitsConfig(
    name="mv_split",
    test_frac=0.2,  # 20% of full dataset held out for test
    folds=3,  # 3-fold cross-validation; each fold's val is 1/3 of the non-test items
    stratify=True,  # Preserve class distribution in each partition
)

task = TaskConfig(
    name="split_military_vehicles",
    workflow="mv_split",
    sources="mv_src",
)

bias_task = TaskConfig(name="bias_military_vehicles", workflow="mv_bias", sources="mv_src")

# Build the pipeline configuration
config = PipelineConfig(
    max_processes=2,
    datasets=[
        HuggingFaceDatasetConfig(name="mv_train", path=str(data_path), task="image_classification"),
    ],
    sources=[
        SourceConfig(name="mv_src", dataset="mv_train"),
    ],
    workflows=[workflow, BiasConfig(name="mv_bias")],
    tasks=[task, bias_task],
)

In [3]:
result = run_task(config, task, cache_dir=Path("./cache"))

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_cache.py:273: UserWarning: `height` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  export(path)


In [4]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [5]:
print(result.report())


  SPLITS
  Workflow:  mv_split (splits)
  Timestamp: 2026-11-07T08:39:41.583664+00:00
  Duration:  12.53s
  Source:    mv_src (mv_train)

  Steps: 6 ran

  SUMMARY
  0
    Class
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]
  1
    Class
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]
  2
    Class
    Stratification .... max deviation 0.1pp (class 'T-72' in split.test)  [ok]

  Health: All checks passed [ok]

  0
  Class Stratification — max deviation 0.1pp (class 'T-72' in split.test)
    Each part's class proportions against `data`'s, in percentage points. Counts
    are labels: one per box on detection data.

    Class           split.train[0]  split.val[0]  split.test      data
    --------------  --------------  ------------  ----------  --------
    T-72                       227           113          84  424 (5%)
    BTR-80                     220           110          83  413 (5%)
    BMP-1                     

In [6]:
# The page styles its whole document, so the docs show it in an iframe of its own.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [7]:
show(result.to_html())

<iframe srcdoc="<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Splits — mv_split</title>
<style>:root { color-scheme: light dark;
 --bg: #ffffff; --card: #ffffff; --soft: #f6f8fa; --ink: #111827; --muted: #6b7280; --rule: #e5e7eb;
 --link: #1d4ed8; --ok: #16a34a; --info: #2563eb; --warning: #d97706; --mark: #dc2626;
 --s0: #2563eb; --s1: #d97706; --s2: #7c3aed; --s3: #16a34a;
 --tag: #eef2ff; --tag-ink: #1e1b4b;
}
@media (prefers-color-scheme: dark) { :root {
 --bg: #0f1115; --card: #171a21; --soft: #1d212a; --ink: #e5e7eb; --muted: #9ca3af; --rule: #2a2f3a;
 --link: #93c5fd; --ok: #22c55e; --info: #60a5fa; --warning: #f59e0b; --mark: #f87171;
 --s0: #60a5fa; --s1: #f59e0b; --s2: #a78bfa; --s3: #22c55e;
 --tag: #232a3d; --tag-ink: #c7d2fe;
} }
body { margin: 0; color: var(--ink); background: var(--bg);
 font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; }
main { max-width: 72rem; margin: 0 auto; padding: 1.5rem; }
a { color: var(--link); }
h1, h2, h3, h4 { margin: 1.25rem 0 0.5rem; line-height: 1.25; }
h1 { font-size: 1.6rem; } h2 { font-size: 1.2rem; } h3 { font-size: 1.05rem; } h4 { font-size: 0.95rem; }
nav.contents { border: 1px solid var(--rule); border-radius: 7px; padding: 0.25rem 1rem; margin-bottom: 1.5rem; }
nav.contents h2 { font-size: 1rem; margin: 0.5rem 0; }
.controls { display: flex; justify-content: flex-end; gap: 0.5rem; margin-bottom: 0.5rem; }
.controls button { font: inherit; font-size: 0.8rem; padding: 0.2rem 0.7rem; cursor: pointer;
 color: var(--ink); background: var(--soft); border: 1px solid var(--rule); border-radius: 4px; }
.report { margin-bottom: 2.5rem; }
.report-head { display: flex; align-items: baseline; gap: 0.75rem; flex-wrap: wrap;
 border-bottom: 2px solid var(--ink); padding-bottom: 0.3rem; }
.report-head h1 { margin: 0; }
dl.fields.provenance { display: grid; grid-template-columns: max-content 1fr; gap: 0.15rem 1rem;
 margin: 0.6rem 0 1.2rem; padding: 0.5rem 0.8rem; background: var(--soft); border-radius: 6px;
 color: var(--muted); font-size: 0.85rem; }
dl.provenance dt { font-weight: 600; color: var(--ink); }
dl.provenance dd { margin: 0; overflow-wrap: anywhere; }
section.section > h2 { border-bottom: 1px solid var(--rule); padding-bottom: 0.2rem; }
details.card, details.panel { background: var(--card); border: 1px solid var(--rule);
 border-left: 4px solid var(--info); border-radius: 7px; padding: 0.3rem 1rem 0.6rem; margin: 0.9rem 0; }
details.card.ok { border-left-color: var(--ok); } details.card.warning { border-left-color: var(--warning); }
details.panel { border-left-color: var(--rule); margin-top: 1.5rem; }
:is(details.card, details.panel) > summary { cursor: pointer; list-style: none; }
:is(details.card, details.panel) > summary::-webkit-details-marker { display: none; }
:is(details.card, details.panel) > summary h2 { display: inline; font-size: 1.1rem; margin: 0; }
:is(details.card, details.panel) > summary::before { content: "▸"; color: var(--muted); margin-right: 0.4rem; }
:is(details.card, details.panel)[open] > summary::before { content: "▾"; }
.brief { font-weight: normal; color: var(--muted); margin-left: 0.5rem; }
.badge { font-size: 0.72rem; font-weight: 700; letter-spacing: 0.02em; border-radius: 1rem; padding: 0.05rem 0.55rem;
 margin-left: 0.5rem; color: #fff; background: var(--info); vertical-align: middle; }
.badge.ok { background: var(--ok); } .badge.warning { background: var(--warning); }
.badge.failed { background: var(--mark); }
code, pre { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "DejaVu Sans Mono", monospace; }
code { font-size: 0.9em; background: var(--soft); padding: 0 0.2em; border-radius: 3px; }
pre { background: var(--soft); padding: 0.75rem; overflow-x: auto; font-size: 0.85rem; }
pre code { background: none; padding: 0; }
table { border-collapse: collapse; margin: 0.5rem 0; font-size: 0.85rem; fon

In [8]:
indices = result.steps["split"].details["indices"]
test = indices["test"]
print(f"Folds: {list(indices['train'])}")
print(f"Test indices: {len(test)} (first ten: {test[:10]})")

Folds: ['0', '1', '2']
Test indices: 1565 (first ten: [69, 70, 71, 72, 73, 74, 75, 76, 77, 78])


In [9]:
import polars as pl

rows = [
    {"fold": fold, "train": len(indices["train"][fold]), "val": len(indices["val"][fold])} for fold in indices["train"]
]
print(pl.DataFrame(rows))
print(f"\nTest (shared across folds): {len(test)} samples")

shape: (3, 3)
┌──────┬───────┬──────┐
│ fold ┆ train ┆ val  │
│ ---  ┆ ---   ┆ ---  │
│ str  ┆ i64   ┆ i64  │
╞══════╪═══════╪══════╡
│ 0    ┆ 4172  ┆ 2086 │
│ 1    ┆ 4172  ┆ 2086 │
│ 2    ┆ 4172  ┆ 2086 │
└──────┴───────┴──────┘

Test (shared across folds): 1565 samples


In [10]:
# Verify no overlap between parts and full coverage per fold
dataset_size = sum(len(indices[part]["0"]) for part in ("train", "val")) + len(test)
test_set = set(test)

for fold in indices["train"]:
    train_set = set(indices["train"][fold])
    val_set = set(indices["val"][fold])

    assert train_set.isdisjoint(val_set), f"Fold {fold}: train/val overlap!"
    assert train_set.isdisjoint(test_set), f"Fold {fold}: train/test overlap!"
    assert val_set.isdisjoint(test_set), f"Fold {fold}: val/test overlap!"
    assert len(train_set | val_set | test_set) == dataset_size, f"Fold {fold}: missing indices"

print(f"All {len(indices['train'])} folds verified: no overlap, full coverage of {dataset_size} items.")

All 3 folds verified: no overlap, full coverage of 7823 items.


In [11]:
for item in result.findings:
    print(f"{item.severity:8} {item.title}: {item.brief}")

ok       Class Stratification: max deviation 0.1pp (class 'T-72' in split.test)
ok       Class Stratification: max deviation 0.1pp (class 'T-72' in split.test)
ok       Class Stratification: max deviation 0.1pp (class 'T-72' in split.test)


In [12]:
bias = run_task(config, bias_task, cache_dir=Path("./cache"))
for item in bias.findings:
    print(f"{item.severity:8} {item.title}: {item.brief}")
print(bias.steps["balance"].output.balance)
print(bias.steps["diversity"].output.factors)

/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_chain/_run.py:176: ExperimentalWarning: The 'parity' evaluator is experimental: it may change or be removed in any release without a deprecation cycle.
  warn_if_experimental(step)
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/_chain/_run.py:176: ExperimentalWarning: The 'factor-parity' check is experimental: it may change or be removed in any release without a deprecation cycle.
  warn_if_experimental(step)
/builds/jatic/aria/dataeval-flow/src/dataeval_flow/workflows/_common.py:60: UserWarning: `height` and `width` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"height": [...]} to control this.
  factor_info = metadata.factor_info


/builds/jatic/aria/dataeval-flow/src/dataeval_flow/evaluators/bias/_evaluator.py:92: ExperimentalWarning: 'Parity' is experimental and may change or be removed in any future release without notice.
  return Parity(**dataeval_arguments(config)).evaluate(metadata)
/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/dataeval/bias/_parity.py:478: ExperimentalWarning: 'parity' is experimental and may change or be removed in any future release without notice.
  result = parity(factor_data, axis.values, unit.groups, assume_independent=unit.groups is None)


info     Class Imbalance: 24 classes, 7823 items, imbalance 3.6:1
ok       Shortcut Risk: 2 factors checked
ok       Factor Parity: 2 factors checked
info     Parity Collection Gaps: 12 under-represented cells, 1136 cells too thin to judge
ok       Factor Coverage Gaps: No significant gaps detected
shape: (3, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ cat         ┆ f64      │
╞═════════════╪══════════╡
│ class_label ┆ 1.0      │
│ height      ┆ 0.008153 │
│ width       ┆ 0.009517 │
└─────────────┴──────────┘
shape: (3, 3)
┌─────────────┬─────────────────┬──────────────────┐
│ factor_name ┆ diversity_value ┆ is_low_diversity │
│ ---         ┆ ---             ┆ ---              │
│ cat         ┆ f64             ┆ bool             │
╞═════════════╪═════════════════╪══════════════════╡
│ class_label ┆ 0.957973        ┆ false            │
│ height      ┆ 0.153298        ┆ true             │
│ width       ┆ 0.169882        ┆ true             │
└────

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/dataeval/bias/_parity.py:601: ExperimentalWarning: 'ParityOutput' is experimental and may change or be removed in any future release without notice.
  return ParityOutput(


In [13]:
print(f"Folds:      {len(indices['train'])}")
print(f"Source:     {result.metadata.lineage[0].name} ({result.metadata.lineage[0].items} items)")
for record in result.metadata.lineage[1:4]:
    print(f"{record.name:16} {record.type:6} {record.items} items")

Folds:      3
Source:     data (7823 items)
split.train[0]   kfold  4172 items
split.train[1]   kfold  4172 items
split.train[2]   kfold  4172 items


In [14]:
import json

exported = json.loads(result.export(fmt="json"))

# The same indices, under the split step's details
exported_indices = exported["steps"]["split"]["details"]["indices"]
print(f"Test indices ({len(exported_indices['test'])} samples): {exported_indices['test'][:10]}...")

for fold in exported_indices["train"]:
    print(f"Fold {fold}: train={len(exported_indices['train'][fold])}, val={len(exported_indices['val'][fold])}")

Test indices (1565 samples): [69, 70, 71, 72, 73, 74, 75, 76, 77, 78]...
Fold 0: train=4172, val=2086
Fold 1: train=4172, val=2086
Fold 2: train=4172, val=2086
